In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallHistogram"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#GETTING PRECIP DATA

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

In [ ]:
# #DATA LOADING FOR NCEP/EMC LEVEL IV DATA (*OLD*)

# def ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory, region='conus'):
#     yearmonth = yearmonthdayhour[0:6]
#     inputPath = os.path.join(inputDirectory,yearmonth,f"st4_{region}.{yearmonthdayhour}.01h.grb2")
#     precipData = xr.open_dataset(inputPath, engine="cfgrib")['tp']
#     #Note: data in kg/m^2 = mm since rho = m/V = m/A/h where m/A = 1 ==> h = 1e-3 m = 1 mm
#     return precipData

# def GetAccumulatedPrecipData_LevelIV(ModelData): 
#     #getting inputDirectory
#     inputDirectory = os.path.join(DirectoryManager.dataDirectory,
#                                   f"Observation_Data/{ModelData.region}/StageIV_PrecipData")
#     print(f"reading from {inputDirectory}")

#     #getting yearmonthdayhour list
#     dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
#     start = dt_list[0]
#     target = start + timedelta(hours=12)
    
#     # Find the entry closest to +12 hours
#     closest = min(dt_list, key=lambda x: abs(x - target))
    
#     closest_idx = dt_list.index(closest)
#     times = ModelData.timeStrings[closest_idx:]
#     yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
#                                 for t in times})
    
#     for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
#         if count == 0:
#             precipData_T = ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)
#         else:
#             precipData_T += ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)

#     precipData_T = precipData_T.assign_coords(longitude = precipData_T.longitude - 360)
#     return precipData_T

# precipData_T = GetAccumulatedPrecipData_LevelIV(ModelData_NSSL)
# precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion_Curvilinear(precipData_T, ModelData_NSSL)

In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory


def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData


def GetAccumulatedPrecipData_MRMS_QPE(ModelData): 
    #getting inputDirectory
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start + timedelta(hours=13)
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)
        
        precipData = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if precipData is None:
            continue #skips missing hours
        if precipData_T is None:
            precipData_T = precipData #equates at first timestep
        else:
            precipData_T += precipData
            

    precipData_T = precipData_T.assign_coords(longitude = precipData_T.longitude - 360)
    precipData_T = precipData_T.sortby("latitude")
    return precipData_T

In [ ]:
precipData_T = GetAccumulatedPrecipData_MRMS_QPE(ModelData_NSSL)
precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion(precipData_T, ModelData_NSSL)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def RunCalculations(ModelData, varNames):
    #Gets the last timestep rainnc+rainc and later RemoveFirstTwelveHours removes the spinup rain
    #(Another option is to sum raincv, rainncv separately)
    outputDictionary={}
    
    t = ModelData.Ntime-1
        
    #Loading Data
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _,_, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

    for varName in varNames:
        print(f"Running for {varName}")
        #Subsetting Data

        variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
        
    return variableSubset, variableSubset.latitude, variableSubset.longitude

In [ ]:
def FindIndexAtHourOffset(timeStrings, hourOffset):
    # Convert strings → datetime objects
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeStrings]
    
    # Starting time
    start = dt_list[0]
    target = start + timedelta(hours=hourOffset)
    
    # Find closest timestamp
    closest_dt = min(dt_list, key=lambda x: abs(x - target))
    idx = dt_list.index(closest_dt)
    
    return idx, timeStrings[idx]
    
def RemoveFirstTwelveHours(ModelData, rain_model):
    varName = 'rainnc+rainc'
    t12, _ = FindIndexAtHourOffset(ModelData.timeStrings, 12)
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _,_, _,_] = DataOperator_Class.GetData_Subset(ModelData, t12)
    variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    rain_model -= variableSubset
    return rain_model

In [ ]:
####################################
#CALCULATION

In [ ]:
rain_NSSL, lat,lon = RunCalculations(ModelData=ModelData_NSSL,varNames=['rainnc+rainc'])
rain_TEMPO, _,_ = RunCalculations(ModelData=ModelData_TEMPO,varNames=['rainnc+rainc'])

In [ ]:
rain_NSSL = RemoveFirstTwelveHours(ModelData=ModelData_NSSL, rain_model=rain_NSSL)
rain_TEMPO = RemoveFirstTwelveHours(ModelData=ModelData_TEMPO, rain_model=rain_TEMPO)

In [ ]:
# #Applying Radar Mask
# RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)
# RadarDataMask_interp = (RadarDataMask*1).interp(
#     latitude = precipData_T_Subset.latitude,
#     longitude = precipData_T_Subset.longitude,
#     method="nearest"
# )
# RadarDataMask_interp = RadarDataMask_interp == 1

# rain_NSSL = rain_NSSL.where(RadarDataMask == True) #commenting out since precip data has wider mask
# rain_TEMPO = rain_TEMPO.where(RadarDataMask == True)
# precipData_T_Subset = precipData_T_Subset.where(RadarDataMask_interp == True)

In [ ]:
# def InterpToStageIV(model_var, model_lat, model_lon, stage_lat2d, stage_lon2d): 
#     """
#     Interpolates MPAS (lat1d, lon1d) field to Stage-IV curvilinear grid.

#     Returns DataArray with EXACT Stage-IV dims/coords
#     (no artificial y/x dims).
#     """

#     # 1. Extract numpy values
#     model_array = np.asarray(model_var)

#     # 2. Build MPAS points
#     mpas_lon2d, mpas_lat2d = np.meshgrid(model_lon, model_lat)
#     points = np.column_stack([mpas_lon2d.ravel(), mpas_lat2d.ravel()])
#     values = model_array.ravel()

#     # 3. Target grid (Stage-IV)
#     stage_points = np.column_stack([stage_lon2d.ravel(), stage_lat2d.ravel()])

#     # 4. Interpolate
#     interp_flat = griddata(points, values, stage_points, method="linear")
#     interp2d = interp_flat.reshape(stage_lat2d.shape)

#     # 5. Return DataArray preserving Stage-IV coords
#     da = xr.DataArray(
#         interp2d,
#         dims=precipData_T_Subset.dims,  # <<— IMPORTANT
#         coords={
#             "latitude": (precipData_T_Subset.dims, stage_lat2d),
#             "longitude": (precipData_T_Subset.dims, stage_lon2d)
#         },
#         name=model_var.name if hasattr(model_var, "name") else "interpolated"
#     )

#     return da


# rain_NSSL_interp = InterpToStageIV(
#     rain_NSSL, lat, lon,
#     precipData_T_Subset.latitude.values,
#     precipData_T_Subset.longitude.values
# )

# rain_TEMPO_interp = InterpToStageIV(
#     rain_TEMPO, lat, lon,
#     precipData_T_Subset.latitude.values,
#     precipData_T_Subset.longitude.values
# )


In [ ]:
#Interpolatting model data onto MRMS data
rain_NSSL_interp = rain_NSSL.interp(longitude=precipData_T_Subset.longitude,latitude=precipData_T_Subset.latitude)
rain_TEMPO_interp = rain_TEMPO.interp(longitude=precipData_T_Subset.longitude,latitude=precipData_T_Subset.latitude)
clim = (
    min(rain_NSSL_interp.min().item(), 
        rain_TEMPO_interp.min().item(), 
        precipData_T_Subset.min().item()),

    max(rain_NSSL_interp.max().item(), 
        rain_TEMPO_interp.max().item(), 
        precipData_T_Subset.max().item())
)

In [ ]:
#coloring zeros white (doesn't effect histograms)

rain_NSSL_interp_whited = rain_NSSL_interp.where(rain_NSSL_interp!=0)
rain_TEMPO_interp_whited = rain_TEMPO_interp.where(rain_TEMPO_interp!=0)
precipData_T_Subset_whited = precipData_T_Subset.where(precipData_T_Subset!=0)

#another option
# rain_NSSL_interp_whited = rain_NSSL_interp.where(rain_NSSL_interp>=0.1)
# rain_TEMPO_interp_whited = rain_TEMPO_interp.where(rain_TEMPO_interp>=0.1)
# precipData_T_Subset_whited = precipData_T_Subset.where(precipData_T_Subset>=0.1)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def CreateFigure_2x2(figsize=(10, 10),
                     wspace=0.3, hspace=0.3,
                     left=0.05, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a 2x2 grid of subplots with adjustable layout.
    Returns (fig, axes) where axes is a 2D list [[ax00, ax01], [ax10, ax11]].
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig)

    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])

    axes = [[ax00, ax01],
            [ax10, ax11]]

    return fig, axes

def CreateFigure_2x1Combo(figsize=(10, 10),
                          wspace=0.35, hspace=0.1,
                          left=0.07, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a figure with two subplots on top (map style)
    and one wide subplot spanning the full bottom row.
    
    Layout:
        +-----------+-----------+
        |   ax00    |   ax01    |
        +-----------------------+
        |        ax_bottom      |
        +-----------------------+
    
    Returns (fig, axes) where:
      axes = [[ax00, ax01], [ax_bottom]]
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 0.8])

    # Two map panels on top
    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())

    # One wide histogram panel across bottom
    ax_bottom = fig.add_subplot(gs[1, :])  # spans both columns

    axes = [[ax00, ax01], [ax_bottom]]

    return fig, axes

def CreateFigure_3x1Combo(figsize=(14, 10),
                          wspace=0.25, hspace=0.15,
                          left=0.06, right=0.97, top=0.93, bottom=0.08):
    """
    Layout:
        +-----------+-----------+-----------+
        |   ax00    |   ax01    |   ax02    |
        +-----------------------------------+
        |             ax_bottom              |
        +-----------------------------------+

    Returns (fig, axes) where:
      axes = [[ax00, ax01, ax02], [ax_bottom]]
    """

    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    # 2 rows × 3 columns grid
    gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 0.8])

    # Top row: 3 map panels
    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax02 = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree())

    # Bottom row: spans all 3 columns
    ax_bottom = fig.add_subplot(gs[1, :])  # slice all columns

    axes = [[ax00, ax01, ax02], [ax_bottom]]

    return fig, axes

In [ ]:
#CONTOUR PLOTTING FUNCTION
from mpl_toolkits.axes_grid1 import make_axes_locatable
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# Preload map features once (global)
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def BuildDiscreteColormap(levels, cmap="turbo"):
    cmap_obj = plt.get_cmap(cmap)
    # one color per level interval
    colors = cmap_obj(np.linspace(0, 1, len(levels)-1))
    discrete_cmap = ListedColormap(colors)
    norm = BoundaryNorm(levels, discrete_cmap.N, clip=False)
    return discrete_cmap, norm
def PlotVariable_with_Borders(axis, 
                              variable, varName, lat, lon, multiplier=1, 
                              clim=(None, None), norm=None, cmap="turbo",
                              title=None, units=None,
                              center_colorbar=False, levels=None):

    # --- Compute bounds ---
    vmin = np.nanmin(variable) if clim[0] is None else clim[0]
    vmax = np.nanmax(variable) if clim[1] is None else clim[1]

    # --- Levels ---
    if levels is None:
        num_levels = 19
        levels = np.linspace(vmin, vmax, num_levels)
    else:
        levels = np.asarray(levels)

    matrix = multiplier * variable

    # --- Build discrete colormap + norm ---
    if norm is None:
        discrete_cmap, norm = BuildDiscreteColormap(levels, cmap=cmap)
    else:
         discrete_cmap = plt.get_cmap(cmap)

    # --- DISCRETE PCOLORMESH ---
    im = axis.pcolormesh(
        lon, lat, matrix,
        cmap=discrete_cmap,
        norm=norm,
        shading="nearest",
        transform=ccrs.PlateCarree()
    )

    # --- Map features ---
    axis.add_feature(COAST,  linewidth=1.2, edgecolor="green")
    axis.add_feature(BORDERS, linewidth=0.8, edgecolor="green")
    axis.add_feature(STATES,  linewidth=0.5, edgecolor="green")
    axis.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    axis.add_feature(LAKES, edgecolor="k", facecolor="none")

    # --- Labels / axes ---
    axis.set_extent([lon.min(), lon.max(), lat.min(), lat.max()],
                    crs=ccrs.PlateCarree())
    axis.set_xticks(np.round(np.linspace(lon.min(), lon.max(), 5), 1),
                    crs=ccrs.PlateCarree())
    axis.set_yticks(np.round(np.linspace(lat.min(), lat.max(), 5), 1),
                    crs=ccrs.PlateCarree())

    axis.set_xlabel("Longitude (°E)")
    axis.set_ylabel("Latitude (°N)")

    if title:
        axis.set_title(title)

    return im


In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=150):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallHistogram.png"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def PlotRainContours(axes, rain_NSSL, rain_TEMPO, lat, lon, clim,
                     ModelData_NSSL, ModelData_TEMPO,
                     precipData_T_Subset,
                     num_levels=None,
                     logscale=True):
    """
    Plot spatial rain accumulation maps for NSSL, Stage-IV, and TEMPO,
    all using the MPAS model lat/lon limits.
    """

    # Shared discrete levels for all 3 panels
    if num_levels is None:
        num_levels = 19
    if logscale==True:
        shared_levels = np.logspace(np.log10(max(clim[0],0.1)), np.log10(clim[1]), num_levels)
    else:
        shared_levels = np.linspace(clim[0], clim[1], num_levels)

    # Domain limits
    # lat_min, lat_max = float(lat.min()), float(lat.max())
    # lon_min, lon_max = float(lon.min()), float(lon.max())
    lat_min = float(ModelData_NSSL.latitude.min())
    lat_max = float(ModelData_NSSL.latitude.max())
    lon_min = float(ModelData_NSSL.longitude.min())
    lon_max = float(ModelData_NSSL.longitude.max())
    
    # ------------------------------------------
    # 1. NSSL panel
    # ------------------------------------------
    axis = axes[0][0]
    im = PlotVariable_with_Borders(
        axis,
        rain_NSSL,
        "",
        lat, lon,
        clim=clim,
        levels=shared_levels
    )
    axis.set_title(f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType} rainnc+rainc')
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    axis.set_ylabel("longitude")
    axis.set_xlabel("latitude")

    # ------------------------------------------
    # 2. Stage IV panel (now using same function)
    # ------------------------------------------
    axis = axes[0][1]
    PlotVariable_with_Borders(
        axis,
        precipData_T_Subset.values,
        "",
        precipData_T_Subset.latitude.values,
        precipData_T_Subset.longitude.values,
        clim=clim,
        levels=shared_levels
    )
    axis.set_title("MRMS QPE 01H Pass 2 Accumulated Precip")
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    axis.set_xlabel("latitude")
    axis.set_ylabel("")

    # ------------------------------------------
    # 3. TEMPO panel
    # ------------------------------------------
    axis = axes[0][2]
    PlotVariable_with_Borders(
        axis,
        rain_TEMPO,
        "",
        lat, lon,
        clim=clim,
        levels=shared_levels
    )
    axis.set_title(f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType} rainnc+rainc')
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    axis.set_xlabel("latitude")
    axis.set_ylabel("")

    # ------------------------------------------
    # 4. Shared colorbar
    # ------------------------------------------
    cbar = fig.colorbar(
        im,
        ax=axes[0],         # attach to all first-row axes
        orientation="vertical",
        fraction=0.03,
        pad=0.02
    )
    cbar.set_label("Rainfall (mm)")
    cbar.set_ticks(shared_levels)
    
    from matplotlib.ticker import FuncFormatter
    
    cbar.formatter = FuncFormatter(lambda x, pos: f"{x:.2f}")
    cbar.update_ticks()

In [ ]:
def PlotRainHistograms(axes, rain_NSSL, rain_TEMPO, precipData_T_Subset):
    """
    Plot outlined histograms of accumulated precipitation for NSSL and TEMPO
    on a single combined bottom axis using shared bins and different colors.
    """
    # Define the bottom axis (only one now)
    axis = axes[1][0]

    # Getting valid non-nan data
    valid_NSSL  = rain_NSSL.values.flatten()
    valid_NSSL  = valid_NSSL[~np.isnan(valid_NSSL)]
    
    valid_TEMPO = rain_TEMPO.values.flatten()
    valid_TEMPO = valid_TEMPO[~np.isnan(valid_TEMPO)]
    
    valid_NOAA  = precipData_T_Subset.data.flatten()
    valid_NOAA  = valid_NOAA[~np.isnan(valid_NOAA)]

    # Common bins for fair comparison
    combinedData = np.concatenate([valid_NSSL, valid_TEMPO, valid_NOAA])
    binEdges = np.linspace(combinedData.min(), combinedData.max(), 51)  # 50 bins

    # NSSL histogram (outlined)
    label = f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType}'
    axis.hist(valid_NSSL, bins=binEdges,
              histtype='step', linewidth=1.8, color='steelblue', label=label)

    # NOAAnClimGrid histogram (outlined)
    label = f'MRMS QPE 01H Pass 2'
    axis.hist(valid_NOAA, bins=binEdges,
              histtype='step', linewidth=1.8, color='green', label=label)

    # TEMPO histogram (outlined)
    label = f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType}'
    axis.hist(valid_TEMPO, bins=binEdges,
              histtype='step', linewidth=1.8, color='darkorange', label=label)

    # Log scale, labels, and limits
    axis.set_yscale("log")
    axis.set_xlim(left=0, right=np.nanmax(combinedData))
    axis.set_xlabel('rainnc + rainc (accumulated precipitation) (mm)')
    axis.set_ylabel('count')

    # Add legend
    axis.legend(frameon=False, fontsize=12)


In [ ]:
####################################
#PLOTTING

In [ ]:
fig, axes = CreateFigure_3x1Combo(figsize=(15,10))

PlotRainContours(axes, rain_NSSL_interp_whited, rain_TEMPO_interp_whited, precipData_T_Subset.latitude.values, precipData_T_Subset.longitude.values, clim,
                 ModelData_NSSL, ModelData_TEMPO,
                 precipData_T_Subset_whited)

PlotRainHistograms(axes, rain_NSSL_interp, rain_TEMPO_interp, precipData_T_Subset)

SaveFigure(fig, ModelData1=ModelData_NSSL, ModelData2=ModelData_TEMPO)

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS
plotting = False #keep false when job array is running
# plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
               extension="png"):
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(outputPlottingDirectory, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{dataType}.{extension}"
    )
    return inputFilePath

def GetFilePaths():
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    filePaths = GetFilePaths()
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 4),
                                                     wspace=0.01,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=outputPlottingDirectory,fileName=dataType)